# DAF-10: Move the Table Build into `clean.py`, With Tests

## The story

In DAF-09 Asha cleaned the data and built the daily table. It works. But
the code lives **inside a notebook**.

Soon (Phase 8) the forecast becomes a **live service**. Every day at 18:00
it must build **today's row** and ask the model for tomorrow's forecast.

Her friend Ravi says: *"Easy. I'll copy the notebook code into the service."*

Ravi's copy looks the same, but it has **one tiny difference**. To find
"today", it asks the computer for the date. The server runs on **UTC time**,
not India time.

- In Delhi, UTC midnight is **05:30 in the morning**.
- So Ravi's "today" starts at 05:30, not at 00:00.
- The hours from 00:00 to 05:30 are silently left out.

Nothing crashes. No error. No alert. The forecast just becomes a bit wrong,
and nobody knows why.

**The picture below shows what really happens, on one real day from our own
data.**

### The problem in one picture: one real day, 1 March 2026

Read it from top to bottom. Every number is real, from R K Puram.

![Training/serving skew on 1 March 2026](figures/daf10_intro_skew_walkthrough.png)

| Panel | What happens | The real numbers |
|---|---|---|
| ① | **Two clocks.** The notebook's day starts at 00:00 India time. Ravi's copy starts its day at UTC midnight, which is 05:30 in Delhi. | The forecast runs at 18:00 IST (12:30 UTC). Ravi's copy misses 00:00–05:30. |
| ② | **Training.** The model learned from `pm25_until_17` = the average of the hours **00 to 17**. | 17 hours with a reading (00:00 was empty) → **83.6** |
| ③ | **Serving with Ravi's copy.** Same readings, but only hours **06 to 17**. The night hours are the dirtiest, and they're the ones that get dropped. | 01–05 were 132, 102, 97, 110, 80, all dropped. 12 hours → **75.1** |
| ④ | **Same model, two inputs.** The model is the DAF-07 model, unchanged. Only the input number is different. | 83.6 → forecast **92.0** (≥ 91, warning ✓). 75.1 → forecast **88.0** (< 91, no warning ✗). The real Monday was **100**: Poor air. |
| ⑤ | **All 145 test days.** The normal score (MAE) hardly moves, so nobody notices. But the warnings Asha cares about drop by half. | MAE 15.5 → 15.4. Poor days warned: **6 → 3** (of 17). |
| ⑥ | **The fix.** Keep the code in **one file**, `clean.py`. The notebook and the service both `import` it. | Same code → same number (83.6) → the model gets what it was trained on. |

### The words to remember

- **Training/serving skew:** the model is trained with numbers made one way,
  but in real use it gets numbers made a slightly different way. The model
  can't tell. It just gives worse answers.
- **Why it's dangerous:** there's no error. The average score can even look
  fine (panel ⑤). Only the important cases go wrong.
- **The fix:** one copy of the code, used by both. Like a shared library jar
  at work: you don't copy business logic into two microservices, you put it
  in one library and both services depend on it.
- **Why tests too:** if someone changes `clean.py` later (for example, to
  use UTC), a test fails straight away, before the model ever sees a wrong
  number.

## 0. What this ticket asks

### Before and after

```text
BEFORE (DAF-09)                          AFTER (DAF-10)

09_clean.ipynb                           src/delhi_air/clean.py   ← the only copy
  the table code lives here                    │
                                               │  import
live service (later)                           ├──► 10_clean_module.ipynb
  a second copy  ✗ can drift                   ├──► 09_clean.ipynb
                                               └──► live service (Phase 8)

                                         tests/test_clean.py      ← checks every rule
```

### Done means

| # | Criterion (in plain words) | How we prove it |
|---|---|---|
| 1 | `pytest` passes, with at least 5 tests | Every step adds its own tests |
| 2 | One command builds the table: `python -m delhi_air.clean --station 17` writes `data/processed/daily_17.parquet` | Step 6 |
| 3 | The module's table is **exactly** the same as the DAF-09 table | Step 7 compares them in code |
| 4 | No notebook has its own copy of the table code any more | Step 7 changes `09_clean.ipynb` to import |
| 5 | `git status` shows code and tests, no data | Final check |

### The plan: one function per step

Each step moves one piece of DAF-09 into `clean.py`, adds its tests, and
checks the result here.

| Step | Function | In one line |
|---|---|---|
| 1 | `load_raw()` | read the raw files, keep PM2.5 |
| 2 | `to_hourly()` | 4 readings per hour → 1 number per hour |
| 3 | `apply_cleaning()` | the DAF-09 fixes: remove the broken week, fill 1–2 hour holes |
| 4 | `to_daily()` | 24 hours → 1 row per **India** day, with the 18-hour rule |
| 5 | `add_target()` | put tomorrow's average on today's row |
| 6 | `build_daily_table()` + command | all of the above in one call, saved as parquet |
| 7 | — | compare with DAF-09, update `09_clean.ipynb`, final checks |

> **Note:** the ticket lists the cleaning before the hourly step. But all
> the DAF-09 fixes work on **hours**, so we make hours first (Step 2), then
> clean (Step 3). That's the same order DAF-09 really used.

## Step 1. `load_raw()`: raw files → 15-minute PM2.5 readings

### 😖 The problem

In DAF-09 loading was three notebook lines: list the files, `concat` them,
keep `pm25`. Short, but it hides three rules the service must follow **the
same way**:

1. Keep **only** `pm25` rows. Each file also has `pm10`, `no2`, wind and more.
2. Keep the **`+05:30` offset** from the file. Drop it, and every later "day"
   and "hour 17" is wrong.
3. Sort **oldest first**. Later steps (holes, rolling means, "tomorrow") all
   assume time order.

### 💡 The fix: one function, used everywhere

```text
data/raw/openaq/locationid=17/          load_raw(17, raw_root)
  year=2025/month=02/                   ────────────────────────
    location-17-20250219.csv.gz  ──►    1. read every *.csv.gz
    location-17-20250220.csv.gz  ──►    2. keep parameter == "pm25"
    ... 554 files                ──►    3. parse datetime, KEEP +05:30
                                        4. sort oldest first
                                             │
                                             ▼
                                   datetime                    value
                                   2025-02-19 02:00+05:30      ...
                                   ... 44,139 rows
```

Two small design choices in the function:

- **`raw_root` is a parameter**, not a hard-coded path. The notebook passes
  the real folder. A test passes a temporary folder with two tiny files.
- **No files → an error**, not an empty table. An empty table would flow
  through every step and come out as an empty forecast, with no error
  anywhere. Failing loudly is the safer default for a service.

### The code, in `src/delhi_air/clean.py`

```python
def load_raw(location_id: int, raw_root: Path) -> pd.DataFrame:
    station_dir = Path(raw_root) / f"locationid={location_id}"
    files = sorted(station_dir.rglob("*.csv.gz"))
    if not files:
        raise FileNotFoundError(f"no raw files under {station_dir}")

    raw_all = pd.concat([pd.read_csv(f, compression="gzip") for f in files], ignore_index=True)
    pm25 = raw_all[raw_all["parameter"] == "pm25"].copy()
    pm25["datetime"] = pd.to_datetime(pm25["datetime"])      # keeps +05:30
    return pm25[["datetime", "value"]].sort_values("datetime").reset_index(drop=True)
```

### What this step should print

| Check | Expected (same as DAF-09 Step 1) |
|---|---|
| PM2.5 rows loaded | 44,139 |
| timezone | UTC+05:30 |
| first timestamp | 2025-02-19 … +05:30 |
| last timestamp | 2026-09-12 (early morning) … +05:30 |

In [1]:
# Step 1a: setup
# autoreload: when clean.py changes, the notebook picks up the new code
# without restarting the kernel. Useful while we move functions over step by step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd

from delhi_air.clean import load_raw

project_root = Path.cwd().parent
raw_root = project_root / "data/raw/openaq"
interim_dir = project_root / "data/interim"
STATION = 17

In [2]:
# Step 1b: load with the module, and check against the DAF-09 numbers
pm25_raw = load_raw(STATION, raw_root)

print("PM2.5 rows:", len(pm25_raw))
print("Timezone:", pm25_raw["datetime"].dt.tz)
print("First timestamp:", pm25_raw["datetime"].min())
print("Last timestamp :", pm25_raw["datetime"].max())

assert len(pm25_raw) == 44_139, "row count differs from DAF-09"
assert str(pm25_raw["datetime"].dt.tz) == "UTC+05:30", "the IST offset was lost"
assert pm25_raw["datetime"].is_monotonic_increasing, "not sorted oldest first"
print("\n✅ Same readings as DAF-09 Step 1")
pm25_raw.head()

PM2.5 rows: 44139
Timezone: UTC+05:30
First timestamp: 2025-02-19 01:45:00+05:30
Last timestamp : 2026-09-12 00:00:00+05:30

✅ Same readings as DAF-09 Step 1


,datetime,value
0,2025-02-19 01:45:00+05:30,123.0
1,2025-02-19 02:00:00+05:30,97.0
2,2025-02-19 02:15:00+05:30,97.0
3,2025-02-19 02:30:00+05:30,97.0
4,2025-02-19 02:45:00+05:30,97.0


### Step 1 tests: a tiny rehearsal before trusting the real data

We have already called `load_raw()` with the real station data:

```text
44,139 real readings
        │
        ▼
   load_raw()
        │
        ▼
large table
```

That proves the function works **today** with the current files.

But it does not prove that it will keep working tomorrow.

Someone could accidentally remove the sorting, timezone handling, or PM2.5 filter. The real dataset is too large to check by hand every time.

---

### The idea: build a tiny fake station

Instead of using 44,139 rows, the test creates only five rows:

```text
Fake files created by the test

file 1                         file 2
─────────────────────          ─────────────────────
23:45  pm25   80               00:15  pm25   95
23:45  no2    40                00:00  pm25   90
                               00:00  pm10  300
```

Before running the function, we already know the correct answer:

```text
Keep only PM2.5
────────────────
23:45  → 80
00:15  → 95
00:00  → 90

Sort oldest first
─────────────────
23:45  → 80
00:00  → 90
00:15  → 95
```

So the expected result is:

```text
datetime                         value
─────────────────────────────    ─────
2025-02-19 23:45 +05:30          80
2025-02-20 00:00 +05:30          90
2025-02-20 00:15 +05:30          95
```

---

### What the test is actually doing

```text
┌──────────────────────────────────────────────┐
│ 1. Arrange: create tiny fake input             │
│    5 rows, including bad/unwanted rows         │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│ 2. Act: call the real production function     │
│    pm25 = load_raw(17, temporary_folder)      │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│ 3. Assert: compare actual output with answer  │
│    columns, values, timezone, and order       │
└──────────────────────────────────────────────┘
```

This is the same pattern as a small JUnit test:

| Test stage | Meaning |
|---|---|
| **Arrange** | Prepare the input |
| **Act** | Call the real code |
| **Assert** | Check the result |

The test does **not** create a second version of `load_raw()`. It calls the real function from `clean.py`.

---

### Why each fake row exists

| Fake row | What it checks |
|---|---|
| `pm25 = 80` | A valid PM2.5 row is kept |
| `no2 = 40` | Other pollutants are removed |
| `pm10 = 300` | Other pollutants are removed |
| `00:15 = 95` before `00:00 = 90` | The function really sorts the output |
| `+05:30` timestamps | The India timezone is preserved |
| Two separate files | Every file is read, not only the first one |

The test data is deliberately small but slightly tricky. Each row has a job.

---

### What happens if `load_raw()` breaks?

Suppose someone removes the sorting:

```text
Actual result:   [80, 95, 90]
Expected result: [80, 90, 95]
```

The assertion fails immediately:

```text
❌ Test failed
Expected the readings to be oldest first,
but the function returned them in the wrong order.
```

That is useful because the failure tells us exactly which promise was broken.

---

### The second test: no files must fail loudly

The test also checks this situation:

```text
station 999
    │
    └── no files
          │
          ▼
      load_raw()
          │
          ▼
   FileNotFoundError ✅
```

An empty station should not quietly produce an empty table:

```text
❌ Dangerous behaviour:
no files → empty table → no forecast → nobody notices
```

The safer behaviour is:

```text
✅ Safe behaviour:
no files → clear error → problem is noticed immediately
```

So Step 1 is not testing the quality of Delhi's air. It is testing the promises made by `load_raw()`:

1. read all files,
2. keep only PM2.5,
3. preserve India time,
4. sort oldest first,
5. fail clearly when there are no files.

### The tests for Step 1: what a test is, and why we need one

#### 😖 The problem

The check above used the **real** data (44,139 rows). That's good for this
notebook, but it isn't enough on its own:

- **You can't check 44,139 rows by eye.** If the count were 44,137, which 2
  rows went missing, and why?
- **`data/` isn't in git** (it's in `.gitignore`). On a new laptop, or in a
  CI build, there's no real data, so a check that needs it can't run.
- **Next month someone changes `clean.py`.** Who reruns this notebook to
  notice? Nobody.

#### 💡 The fix: a unit test on a tiny, hand-made input

A **test** is a small function that:

1. builds an input **so small that you know the right answer in your head**,
2. calls the real function (`load_raw`),
3. checks the answer with `assert`. If the check is false, the test fails.

You already know this from Java. It's the same thing, with different names:

| Java (JUnit) | Python (pytest) | What it does |
|---|---|---|
| `@Test void loadsPm25()` | `def test_load_raw_...():` | one test |
| `assertEquals(expected, actual)` | `assert actual == expected` | one check |
| `assertThrows(X.class, ...)` | `with pytest.raises(X):` | "this must throw an error" |
| `@TempDir Path dir` | `tmp_path` | a fresh empty folder, deleted afterwards |
| `mvn test` | `pytest` | run every test |

We wrote **2 tests** for `load_raw`. The picture below follows the first
one from start to end, then shows the second one.

### Step 1 tests in one picture

![Step 1 tests walkthrough](figures/daf10_step1_tests_walkthrough.png)

| Panel | What happens | The example |
|---|---|---|
| ① | **Make a fake station.** The test writes 2 tiny files into `tmp_path`, in the same folder shape and column shape as the real OpenAQ files. | 5 rows: three `pm25`, one `no2`, one `pm10`. The second file is written **newest first on purpose**, to check that `load_raw` really sorts. |
| ② | **Call the real `load_raw`.** It reads both files and keeps only `pm25`. | `no2` (40) and `pm10` (300) are dropped. |
| ③ | **It sorts oldest first** and keeps 2 columns. | 80, 95, 90 → **80, 90, 95**. Each time still ends in `+05:30`. |
| ④ | **4 checks.** Each one compares with the answer we worked out by hand. | columns · values `[80, 90, 95]` · timezone `UTC+05:30` · first reading = 23:45 in Delhi |
| ⑤ | **Second test.** A station with no files must **stop with an error**. | `load_raw(999, …)` → `FileNotFoundError` → test passes. |

**Why the second test matters:** if `load_raw` returned an empty table
instead, the next steps would quietly make an empty daily table, and the
service would send no forecast, with no error anywhere. Failing loudly
means someone sees the problem straight away.

### The test code, part by part

The file is `tests/test_clean.py`. It has 3 parts.

**Part 1: a helper that writes a tiny raw file**

```python
def write_raw_file(folder, name, rows):
    folder.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame(rows, columns=["location_id", "datetime", "parameter", "value"])
    frame.to_csv(folder / name, index=False, compression="gzip")
```

- It makes the folder, turns our few rows into a table, and saves it as
  `.csv.gz`, just like a real OpenAQ file.
- It's not a test itself (its name doesn't start with `test_`). It just
  saves us from repeating these 3 lines.

**Part 2: test 1, the happy path**

```python
def test_load_raw_keeps_only_pm25_sorted_and_in_ist(tmp_path):
    station = tmp_path / "locationid=17" / "year=2025" / "month=02"
    write_raw_file(station, "location-17-20250219.csv.gz", [
        (17, "2025-02-19T23:45:00+05:30", "pm25", 80.0),
        (17, "2025-02-19T23:45:00+05:30", "no2", 40.0),
    ])
    write_raw_file(station, "location-17-20250220.csv.gz", [
        (17, "2025-02-20T00:15:00+05:30", "pm25", 95.0),    # newest first, on purpose
        (17, "2025-02-20T00:00:00+05:30", "pm25", 90.0),
        (17, "2025-02-20T00:00:00+05:30", "pm10", 300.0),
    ])

    pm25 = load_raw(17, tmp_path)                            # the REAL function

    assert list(pm25.columns) == ["datetime", "value"]
    assert pm25["value"].tolist() == [80.0, 90.0, 95.0]
    assert str(pm25["datetime"].dt.tz) == "UTC+05:30"
    assert pm25["datetime"].iloc[0] == pd.Timestamp("2025-02-19 23:45", tz="Asia/Kolkata")
```

Every test has the same 3 parts: **arrange → act → check**.

| Part | Lines | Meaning |
|---|---|---|
| Arrange | the two `write_raw_file` calls | build the fake station (panel ①) |
| Act | `pm25 = load_raw(17, tmp_path)` | run the real code, once (panels ② and ③) |
| Check | the 4 `assert` lines | compare with the answer we know by hand (panel ④) |

Why these exact rows?

- **`no2` and `pm10` are in there** so we can prove they get dropped.
- **Two files**, so we prove `load_raw` reads **all** files, not just one.
- **23:45 and 00:00 sit on either side of midnight.** Midnight is where the
  day changes, so it's where a timezone mistake would show up.
- **95 is written before 90**, so the only way to get `[80, 90, 95]` is to
  really sort.

**What the 2 timezone checks catch**

| If someone changes `load_raw` to… | Check 3 (`tz`) | Check 4 (23:45 Delhi) |
|---|---|---|
| keep `+05:30` (correct) | ✅ passes | ✅ passes |
| convert times to UTC | ❌ fails: tz is `UTC` | ✅ passes (same moment) |
| throw the `+05:30` away | ❌ fails: no tz | ❌ fails |
| read 23:45 as if it were UTC | ❌ fails | ❌ fails: wrong moment |

So together they guard against the Ravi bug from the intro.

**Part 3: test 2, the error path**

```python
def test_load_raw_fails_loudly_when_station_has_no_files(tmp_path):
    with pytest.raises(FileNotFoundError):
        load_raw(999, tmp_path)
```

- `tmp_path` is empty, so station 999 has no files.
- `with pytest.raises(FileNotFoundError):` means **"the code inside must
  raise this error"**. If it raises it, the test passes. If it returns
  normally (for example, an empty table), the test fails.

### How pytest finds and runs the tests

- It looks in files named `test_*.py` and runs every function named `test_*`.
- **You never create `tmp_path` yourself.** Because the test has an argument
  called `tmp_path`, pytest makes a fresh empty folder and passes it in. It's
  like Spring injecting a bean by its name.
- The result is `PASSED` or `FAILED` for each test. When a test fails,
  pytest prints the exact line and both values, for example:

```text
    assert pm25["value"].tolist() == [80.0, 90.0, 95.0]
E   assert [80.0, 95.0, 90.0] == [80.0, 90.0, 95.0]      ← someone removed the sort
```

Now run them:

In [3]:
# Step 1c: run the tests
import sys
!cd {project_root} && {sys.executable} -m pytest tests/test_clean.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.14, pytest-8.3.3, pluggy-1.6.0 -- /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast
configfile: pyproject.toml
plugins: anyio-4.15.1
collecting ... 


collected 19 items                                                             

tests/test_clean.py::test_load_raw_keeps_only_pm25_sorted_and_in_ist 

PASSED [  5%]
tests/test_clean.py::test_load_raw_fails_loudly_when_station_has_no_files PASSED [ 10%]
tests/test_clean.py::test_to_hourly_averages_readings_inside_the_same_hour PASSED [ 15%]
tests/test_clean.py::test_to_hourly_leaves_a_gap_as_nan_instead_of_skipping_it PASSED [ 21%]
tests/test_clean.py::test_to_hourly_does_not_require_sorted_input PASSED [ 26%]
tests/test_clean.py::test_fill_short_holes_fills_up_to_the_limit_and_no_further PASSED [ 31%]
tests/test_clean.py::test_remove_days_blanks_only_the_named_range PASSED [ 36%]
tests/test_clean.py::test_apply_cleaning_removes_the_broken_week_before_filling_holes PASSED [ 42%]
tests/test_clean.py::test_apply_cleaning_still_fills_short_holes_outside_the_broken_week PASSED [ 47%]
tests/test_clean.py::test_apply_cleaning_does_not_change_its_input PASSED [ 52%]
tests/test_clean.py::test_to_daily_17_hours_is_invalid_18_is_valid PASSED [ 57%]
tests/test_clean.py::test_to_daily_day_boundary_is_ist_midnight_not_utc PASSED [ 63%]
tests/test_

PASSED [ 94%]
tests/test_clean.py::test_main_fails_loudly_when_the_station_has_no_raw_files PASSED [100%]

============================== 19 passed in 0.40s ==============================


### What Step 1 shows

- `load_raw()` gives back the **same 44,139 readings** as DAF-09, still in
  IST, oldest first.
- **2 tests pass.** Neither needs the real data: each builds its own tiny
  files in a temporary folder.
- The notebook has **no loading code of its own** any more. It only calls
  the module. The Phase 8 service will make the same call.

**Next: Step 2, `to_hourly()`.**

## Step 2: `to_hourly()` — many 15-minute readings become one hourly table

### 🎬 How we got here

In Step 1, `load_raw()` gave us individual PM2.5 readings:

```text
02:00 → 80
02:15 → 90
02:30 → 100
02:45 → 110
```

The forecast should not work with four separate readings for the same hour.

---

### 😖 The problem

We need to answer:

> “What was the average PM2.5 during each hour?”

But some readings may be missing:

```text
14:00 → 80
14:15 → 90
14:30 → missing
14:45 → 100
```

Also, there may be a complete missing hour:

```text
14:00 → 80
15:00 → no readings
16:00 → 100
```

If we simply remove missing hours, the program cannot tell the difference between:

```text
15:00 was empty
```

and:

```text
15:00 never existed
```

That difference matters to the next step, which fills only small holes.

---

### 💡 The idea

`to_hourly()` creates a timeline with **every hour present**.

Then it calculates the average of the readings belonging to each hour.

```text
15-minute readings                 Hourly result
──────────────────                 ─────────────
14:00 → 80                         14:00 → 90
14:15 → 90              ───────►
14:30 → missing                     15:00 → NaN
14:45 → 100                        16:00 → 100
                                   
15:00 → no reading
16:00 → 100
```

`NaN` means:

> “This hour exists, but no PM2.5 reading was available.”

It does **not** mean zero pollution.

---

### 🔍 What `to_hourly()` actually does

```text
15-minute PM2.5 readings
             │
             ▼
┌──────────────────────────────┐
│ Put readings into hour buckets│
│ 14:00, 14:15, 14:45 → 14:00 │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│ Calculate each hour's average │
│ (80 + 90 + 100) / 3 = 90     │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│ Keep empty hours in the index │
│ 15:00 → NaN                   │
└──────────────────────────────┘
```

The output is no longer a row for every reading.

It is a time series with **one row for every hour**:

```text
Input: 15-minute readings

14:00 → 80
14:15 → 90
14:45 → 100
16:00 → 100


Output: hourly readings

14:00 → 90.0
15:00 → NaN
16:00 → 100.0
```

---

### Three small tests prove the important rules

#### Test 1: average readings inside one hour

```text
Input:

14:00 → 80
14:15 → 90
14:45 → 100

Calculation:

(80 + 90 + 100) / 3
= 270 / 3
= 90

Expected output:

14:00 → 90.0
```

This proves that the function calculates an hourly average.

---

#### Test 2: keep a completely empty hour

```text
Input:

14:00 → 80
15:00 → nothing
16:00 → 100
```

The expected output is:

```text
14:00 → 80.0
15:00 → NaN
16:00 → 100.0
```

The important point is that **15:00 remains visible**.

```text
Wrong:

14:00 → 80.0
16:00 → 100.0

Correct:

14:00 → 80.0
15:00 → NaN       ← the hole is visible
16:00 → 100.0
```

The next cleaning step needs this visible hole so it can decide whether to fill it.

---

#### Test 3: input order should not matter

The input might arrive newest first:

```text
Input order:

14:45 → 100
14:15 → 90
14:00 → 80
```

The result must still be:

```text
14:00 → 90.0
```

This proves that `to_hourly()` does not blindly trust the order of its input.

---

### The complete visual picture

```text
┌──────────────────────────────────────┐
│ Step 1 output: raw 15-minute readings│
│                                      │
│ 14:00 → 80                           │
│ 14:15 → 90                           │
│ 14:45 → 100                          │
│ 16:00 → 100                          │
└──────────────────┬───────────────────┘
                   │
                   │ to_hourly()
                   ▼
┌──────────────────────────────────────┐
│ Step 2 output: one row per hour      │
│                                      │
│ 14:00 → 90.0   average of 3 readings │
│ 15:00 → NaN     empty hour preserved  │
│ 16:00 → 100.0  average of 1 reading  │
└──────────────────────────────────────┘
```

So Step 2 is **not cleaning the data yet**.

It is only changing the time resolution:

```text
15-minute data  ───────►  hourly data
many rows per hour      one row per hour
```

The real dataset confirms:

```text
13,680 total hourly slots
12,165 hours with a value
 1,515 empty hours kept as NaN
```

The three small tests prove that those numbers are not accidental:

1. readings are averaged correctly,
2. empty hours remain visible,
3. input order does not affect the result.

In [4]:
# Step 2a: hourly averages with the module, checked against the DAF-09 numbers
from delhi_air.clean import to_hourly

hourly = to_hourly(pm25_raw)

print("Hourly slots:", len(hourly))
print("Hours with data:", hourly.notna().sum())
print("Empty hours:", hourly.isna().sum())

assert len(hourly) == 13_680, "hourly slot count differs from DAF-09"
assert hourly.notna().sum() == 12_165, "hours-with-data differs from DAF-09"
assert hourly.isna().sum() == 1_515, "empty-hour count differs from DAF-09"
assert hourly.index.is_monotonic_increasing, "not sorted oldest first"
print("\n✅ Same hourly averages as DAF-09 Step 3.1 / 3.2")
hourly.head()

Hourly slots: 13680
Hours with data: 12165
Empty hours: 1515

✅ Same hourly averages as DAF-09 Step 3.1 / 3.2


datetime
2025-02-19 01:00:00+05:30    123.0
2025-02-19 02:00:00+05:30     97.0
2025-02-19 03:00:00+05:30     88.0
2025-02-19 04:00:00+05:30    114.0
2025-02-19 05:00:00+05:30     67.0
Freq: h, Name: value, dtype: float64

### Step 2 tests: the two rules that don't show up in one number

`len(hourly) == 13680` proves `to_hourly()` works today, on today's file.
It does not prove the two rules above — a gap stays a gap instead of
vanishing, and unsorted input still comes out right. Those need a tiny,
hand-checked input, same idea as Step 1.

**Test 1 — averaging.** Three readings inside one hour (14:00, 14:15,
14:45 = 80, 90, 100) must collapse to one row, `14:00 → 90.0`.

**Test 2 — the gap stays a gap.** Give it a reading at 14:00 and one at
16:00, nothing at 15:00. The result must still have **3** rows —
14:00, 15:00, 16:00 — with 15:00 equal to `NaN`, not missing from the
index. This is the rule Step 3's hole-filling depends on: it counts holes
by walking the index, so a bucket that silently disappeared would never be
seen as a hole at all.

**Test 3 — order doesn't matter.** The same two readings from Test 1,
written newest-row-first. `to_hourly()` must still return `90.0`, proving
it sorts internally rather than trusting the caller.

In [5]:
# Step 2b: run the tests
import sys
!cd {project_root} && {sys.executable} -m pytest tests/test_clean.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.14, pytest-8.3.3, pluggy-1.6.0 -- /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast
configfile: pyproject.toml
plugins: anyio-4.15.1
collecting ... 


collected 19 items                                                             

tests/test_clean.py::test_load_raw_keeps_only_pm25_sorted_and_in_ist PASSED [  5%]
tests/test_clean.py::test_load_raw_fails_loudly_when_station_has_no_files PASSED [ 10%]
tests/test_clean.py::test_to_hourly_averages_readings_inside_the_same_hour PASSED [ 15%]
tests/test_clean.py::test_to_hourly_leaves_a_gap_as_nan_instead_of_skipping_it PASSED [ 21%]
tests/test_clean.py::test_to_hourly_does_not_require_sorted_input 

PASSED [ 26%]
tests/test_clean.py::test_fill_short_holes_fills_up_to_the_limit_and_no_further PASSED [ 31%]
tests/test_clean.py::test_remove_days_blanks_only_the_named_range PASSED [ 36%]
tests/test_clean.py::test_apply_cleaning_removes_the_broken_week_before_filling_holes PASSED [ 42%]
tests/test_clean.py::test_apply_cleaning_still_fills_short_holes_outside_the_broken_week PASSED [ 47%]
tests/test_clean.py::test_apply_cleaning_does_not_change_its_input PASSED [ 52%]
tests/test_clean.py::test_to_daily_17_hours_is_invalid_18_is_valid PASSED [ 57%]
tests/test_clean.py::test_to_daily_day_boundary_is_ist_midnight_not_utc PASSED [ 63%]
tests/test_clean.py::test_to_daily_pm25_until_17_ignores_hours_18_to_23 PASSED [ 68%]
tests/test_clean.py::test_hours_known_at_18_masks_a_fill_that_needs_an_evening_reading PASSED [ 73%]
tests/test_clean.py::test_hours_known_at_18_keeps_a_fill_whose_neighbours_are_earlier_in_the_day PASSED [ 78%]
tests/test_clean.py::test_add_target_is_tomorrows_mean_written_

PASSED [ 94%]
tests/test_clean.py::test_main_fails_loudly_when_the_station_has_no_raw_files PASSED [100%]

============================== 19 passed in 0.31s ==============================


### What Step 2 shows

- `to_hourly()` gives back the **same 13,680 hourly slots** as DAF-09 —
  12,165 with a reading, 1,515 empty — still sorted oldest first.
- **5 tests pass**, 3 of them new: averaging inside an hour, a gap that
  stays a gap instead of disappearing, and input order not mattering.
- The notebook now has **no hourly-averaging code of its own** either. It
  only calls the module, same as Step 1.

**Next: Step 3, `apply_cleaning()` — the DAF-09 decisions, in one fixed
order.**

## Step 3: `apply_cleaning()` — repair only known problems, in the correct order

### 🎬 How we got here

Step 2 gave us one value for every hour:

```text
14:00 → 80
15:00 → NaN
16:00 → 100
```

Some `NaN` values are small, accidental gaps. Other missing periods are caused by a known sensor problem.

We must handle those two situations differently.

---

### 😖 The problem

A missing hour does not always mean the same thing:

```text
Case A: one small gap

14:00 → 80
15:00 → NaN
16:00 → 100
```

This is probably a short missing reading. We can estimate 15:00:

```text
80 ─────────────── 100
       90
14:00   15:00   16:00
```

But now consider the broken sensor week:

```text
04 July ─────────────────────── 11 July
        sensor was unreliable
```

Values from this period must not be used:

```text
03 Jul → 120
04 Jul → 140
05 Jul → 150
06 Jul → 135
07 Jul → 160
08 Jul → 145
09 Jul → 155
10 Jul → 130
11 Jul → 125
12 Jul → 110
```

Those numbers may look real, but the sensor was known to be broken.

If we fill gaps **before** removing this week, we may calculate an estimate using bad sensor readings.

---

### 💡 The idea

`apply_cleaning()` follows a fixed two-step pipeline:

```text
Hourly data
    │
    ▼
┌──────────────────────────────────────────┐
│ Step 1: remove_days()                    │
│ Blank the known broken sensor week       │
│ 04–11 July 2025                          │
└──────────────────┬───────────────────────┘
                   │
                   ▼
┌──────────────────────────────────────────┐
│ Step 2: fill_short_holes()               │
│ Fill only small gaps of 1–2 hours        │
│ using neighbouring valid values          │
└──────────────────┬───────────────────────┘
                   │
                   ▼
             cleaned hourly data
```

The order is important:

```text
✅ Correct:

remove broken week → fill small valid gaps

❌ Wrong:

fill gaps → remove broken week
```

The first step removes values that should never be trusted.

Only after that do we fill short gaps.

---

### 🔍 What this means visually

Before cleaning:

```text
hourly data

normal values     small gap       broken week       small gap
───────●───────●───────○───────●───●───●───●───●───●───────○───────●───────
       │                       │                   │
       │                       │                   │
   trustworthy            sensor unreliable    trustworthy
```

After `remove_days()`:

```text
hourly data

normal values     small gap       broken week removed       small gap
───────●───────●───────○──────────○───○───○───○───○───○───────○───────●───────
                               all values become NaN
```

After `fill_short_holes()`:

```text
cleaned data

normal values     filled gap       broken week stays empty   filled gap
───────●───────●───────●───────────○───○───○───○───○───○───────●───────●───────
                         ↑                                  ↑
                    small gap filled                   small gap filled
```

The broken week remains empty because it is too large and was deliberately removed.

---

### The two cleaning rules

| Rule | What it does | Visual result |
|---|---|---|
| **Fault 7: broken sensor week** | Remove every value from 4–11 July 2025 | A large empty block |
| **Fault 2: short missing hours** | Fill gaps of 1–2 hours using nearby values | Small holes disappear |
| Other five faults | Leave the data unchanged | No visual change |

The five unchanged faults are not ignored accidentally. The project decision is that they represent real air pollution, a valid single-sensor period, or data outside this PM2.5 pipeline.

---

### A small example of interpolation

Suppose we have:

```text
14:00 → 80
15:00 → NaN
16:00 → 100
```

The missing value is halfway between 80 and 100:

```text
80 ─────────────── 100
       halfway
         90
```

So the result becomes:

```text
14:00 → 80
15:00 → 90
16:00 → 100
```

But a three-hour gap is not filled:

```text
14:00 → 80
15:00 → NaN
16:00 → NaN
17:00 → NaN
18:00 → 100
```

This remains:

```text
14:00 → 80
15:00 → NaN
16:00 → NaN
17:00 → NaN
18:00 → 100
```

The rule allows estimates only for short gaps. It does not invent a long section of data.

---

### What the real dataset shows

```text
Before cleaning:

13,680 hourly slots
 1,515 empty hours
```

Then:

```text
1. Remove the broken week
   → 144 additional hours become empty

2. Fill valid short holes
   → 240 empty hours are filled
```

Therefore:

```text
1,515 + 144 - 240 = 1,419 empty hours
```

Visual summary:

```text
Before cleaning       Broken week removed       Short holes filled
─────────────────     ───────────────────       ──────────────────
1,515 empty hours  →  1,659 empty hours      →  1,419 empty hours
```

The final result has fewer empty hours than the original because some small gaps were repaired, while the known broken week remains blank.

---

### The complete visual picture

```text
┌──────────────────────────────────────────────┐
│ Step 2 output: hourly values                  │
│                                              │
│ normal values ── small gaps ── bad week ── ...│
└──────────────────────┬───────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│ remove_days()                                │
│                                              │
│ known bad week becomes NaN                   │
│ normal values are not changed                │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│ fill_short_holes()                           │
│                                              │
│ 1–2 hour gaps are interpolated               │
│ long gaps remain NaN                         │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│ Step 3 output: cleaned hourly values         │
│                                              │
│ trusted values + small repairs + known gap   │
└──────────────────────────────────────────────┘
```

So Step 3 is not trying to make every hour have a value.

It is making two careful decisions:

1. **Remove values known to be unreliable.**
2. **Fill only small gaps where a reasonable estimate is allowed.**

That is why `apply_cleaning()` must always use this order:

```text
remove broken days → fill short holes
```

## Step 3. `apply_cleaning()`: the DAF-09 decisions, in one fixed order

### 😖 The problem

DAF-09 applied its seven decisions across many notebook cells — one cell
per fault, in whatever order they appear on the page. That's the right way
to *explore*: look at Fault 2, look at Fault 7, look at the effect of each
one separately. It is the wrong way to *ship*:

1. **Order changes the answer.** Fault 7 (the broken sensor week) has to
   run **before** Fault 2 (fill short holes) — otherwise a hole-filling
   guess gets made from a reading the sensor never should have produced.
   Nothing stops someone running the cells out of order.
2. **It can't be reused.** Phase 8's live service must clean today's row
   with exactly these two steps, in exactly this order. A second,
   hand-copied version can drift the same way Ravi's UTC clock did in
   Step 0.

Five of the seven DAF-09 faults are "leave it" — real Delhi air, not a
sensor error (Diwali, the single working sensor), or out of scope for a
PM2.5-only model (wind, NOx). Those need no code, only the decision
already written down in `docs/data_faults.md`. Only two faults change a
number:

| Fault | Treatment | Order |
|---|---|---|
| 7. broken sensor, 4–11 Jul 2025 | blank the week | **first** |
| 2. missing hours | fill holes of 1–2 h with a straight line | **second** |

### 💡 The fix

```text
hourly (from Step 2, 1,515 empty of 13,680)
    │
    ▼
┌──────────────── apply_cleaning() ────────────────┐
│  1. remove_days(): blank 4–11 Jul (Fault 7)       │
│  2. fill_short_holes(): fill holes ≤ 2 h (Fault 2)│
│     Faults 1, 3, 4, 5, 6: leave it — no code       │
└─────────────────────────────────────────────────┘
    │
    ▼
hourly_clean (1,419 empty — fewer than before, some holes filled;
              some new gaps from the blanked week)
```

`remove_days()` and `fill_short_holes()` are the two small helpers behind
it — each one fault, each one a function short enough to read in one
breath.

In [6]:
# Step 3a: clean the hourly series, checked against the DAF-09 numbers
from delhi_air.clean import apply_cleaning

hourly_before = hourly.copy()
hourly_clean = apply_cleaning(hourly)

print("Hourly slots:", len(hourly_clean))
print("Empty hours before cleaning:", hourly.isna().sum())
print("Empty hours after cleaning :", hourly_clean.isna().sum())

assert len(hourly_clean) == 13_680, "hourly slot count changed"
assert hourly_clean.isna().sum() == 1_419, "empty-hour count differs from DAF-09"
assert hourly.equals(hourly_before), "apply_cleaning changed its input"

# Diwali is the night the whole project exists for — cleaning must never touch it
assert hourly_clean.max() == 1753.0, "the Diwali peak changed"
assert hourly_clean.idxmax() == hourly.idxmax(), "the Diwali peak moved"

print("\n✅ Same cleaned series as DAF-09 Step 4")
print("✅ Diwali peak still", hourly_clean.max(), "at", hourly_clean.idxmax())

Hourly slots: 13680
Empty hours before cleaning: 1515
Empty hours after cleaning : 1419

✅ Same cleaned series as DAF-09 Step 4
✅ Diwali peak still 1753.0 at 2025-10-21 03:00:00+05:30


### Step 3 tests: proving the *order*, not just the *count*

`1,419` empty hours after cleaning proves `apply_cleaning()` gets the same
answer as DAF-09 **today**, on today's file. It does not prove the one
rule that actually matters here — that Fault 7 runs before Fault 2. A
count that happens to match can hide a reordering that would only show up
on a different week's data.

**`fill_short_holes()` on its own.** A 1-hour hole gets a straight-line
guess; a 3-hour hole (limit is 2) is left exactly as it was — `NaN`, not a
guess stretched further than the decision allows.

**`remove_days()` on its own.** Only the named range goes empty; the hour
right before it is untouched.

**`apply_cleaning()`, the order that matters.** Put a short (fillable)
hole *inside* the broken week. If Fault 2 ran first, it would happily
interpolate across a gap that Fault 7 is about to blank anyway — no harm
done. But if Fault 7 ran first and Fault 2 still ran second, the hole
would have no good neighbour left to guess from, and must stay empty. The
test checks exactly that: a hole inside the broken week stays empty end to
end, while the same size hole *outside* the week still gets filled.

In [7]:
# Step 3b: run the tests
import sys
!cd {project_root} && {sys.executable} -m pytest tests/test_clean.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.14, pytest-8.3.3, pluggy-1.6.0 -- /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast
configfile: pyproject.toml
plugins: anyio-4.15.1
collecting ... 


collected 19 items                                                             

tests/test_clean.py::test_load_raw_keeps_only_pm25_sorted_and_in_ist PASSED [  5%]
tests/test_clean.py::test_load_raw_fails_loudly_when_station_has_no_files 

PASSED [ 10%]
tests/test_clean.py::test_to_hourly_averages_readings_inside_the_same_hour PASSED [ 15%]
tests/test_clean.py::test_to_hourly_leaves_a_gap_as_nan_instead_of_skipping_it PASSED [ 21%]
tests/test_clean.py::test_to_hourly_does_not_require_sorted_input PASSED [ 26%]
tests/test_clean.py::test_fill_short_holes_fills_up_to_the_limit_and_no_further PASSED [ 31%]
tests/test_clean.py::test_remove_days_blanks_only_the_named_range PASSED [ 36%]
tests/test_clean.py::test_apply_cleaning_removes_the_broken_week_before_filling_holes PASSED [ 42%]
tests/test_clean.py::test_apply_cleaning_still_fills_short_holes_outside_the_broken_week PASSED [ 47%]
tests/test_clean.py::test_apply_cleaning_does_not_change_its_input PASSED [ 52%]
tests/test_clean.py::test_to_daily_17_hours_is_invalid_18_is_valid PASSED [ 57%]
tests/test_clean.py::test_to_daily_day_boundary_is_ist_midnight_not_utc PASSED [ 63%]
tests/test_clean.py::test_to_daily_pm25_until_17_ignores_hours_18_to_23 PASSED [ 68%]
tests/test_cl

PASSED [ 94%]
tests/test_clean.py::test_main_fails_loudly_when_the_station_has_no_raw_files PASSED [100%]

============================== 19 passed in 0.31s ==============================


### What Step 3 shows

- `apply_cleaning()` gives back the **same 1,419 empty hours** as DAF-09 —
  1,515 to start, 144 newly blanked by the broken-week fault, 240 filled
  back in by the short-hole fault. The Diwali peak, 1,753 µg/m³, is
  untouched.
- **10 tests pass**, 5 of them new. The important one is not a count — it
  is that a hole inside the broken week stays empty, proving the two
  faults run in the fixed order the ticket requires, not whatever order a
  caller happens to invoke them in.
- Five of the seven DAF-09 faults needed no code at all — the decision to
  leave them alone lives in `docs/data_faults.md`, not duplicated here.

**Next: Step 4, `to_daily()` — rebuild the daily table exactly as DAF-05,
on the cleaned hours.**

## Step 4: `to_daily()` and `add_target()` — hourly data becomes a daily training table

### 🎬 How we got here

Steps 2 and 3 produced cleaned hourly PM2.5 values:

```text
01 Mar 2026

00:00 → 132
01:00 → 102
02:00 → 97
...
17:00 → 84
18:00 → 35
19:00 → 40
...
23:00 → 48
```

The model should not receive 24 separate hourly rows.

It needs one row per India day.

---

### 😖 The problem

A daily row must answer several different questions:

```text
How polluted was the whole day?
How many hours had readings?
Did we have enough hours to trust this day?
What was the average available by 18:00?
What should today's model predict?
```

A simple average is not enough.

For example:

```text
Day A: 24 valid hours → trustworthy
Day B: 17 valid hours → not enough evidence
Day C:  5 valid hours → definitely unreliable
```

If all three days are treated equally, the model may learn from a daily average based on only a few readings.

There is another danger: the model must predict **tomorrow**, not yesterday.

```text
Today's row must contain tomorrow's PM2.5 as its target.
```

---

### 💡 The idea

Step 4 has two separate jobs:

```text
Hourly data
    │
    ▼
┌─────────────────────────────────────────────┐
│ to_daily()                                  │
│                                             │
│ Group hours by India calendar day           │
│ Calculate daily features                     │
│ Decide whether each day is valid             │
└──────────────────┬──────────────────────────┘
                   │
                   ▼
          one row per day
                   │
                   ▼
┌─────────────────────────────────────────────┐
│ add_target()                                │
│                                             │
│ Move tomorrow's daily mean onto today's row │
└─────────────────────────────────────────────┘
                   │
                   ▼
          model training table
```

This is like separating two responsibilities in a backend service:

```text
to_daily()     → build today's facts
add_target()   → attach tomorrow's answer
```

One function builds the input features.

The other function creates the prediction target.

---

### 🔍 What `to_daily()` actually creates

For every India calendar day, it creates:

| Column | Meaning |
|---|---|
| `pm25_mean` | Average PM2.5 across the valid hours of the day |
| `hours` | Number of hourly readings available |
| `pm25_until_17` | Average of hours 00:00–17:00 |
| `valid` | Whether the day has enough readings |

The transformation is:

```text
24 hourly values
        │
        │ group by India date
        ▼
┌────────────────────────────────────┐
│ One daily row                      │
│                                    │
│ pm25_mean       = whole-day mean  │
│ hours           = readings count  │
│ pm25_until_17  = mean by 18:00    │
│ valid           = hours >= 18     │
└────────────────────────────────────┘
```

---

### Visual example: one day becomes one row

Hourly input:

```text
India date: 01 March 2026

Hour       PM2.5
────────────────
00:00      132
01:00      102
02:00       97
03:00      110
04:00       80
05:00       76
...
17:00       84
18:00       35
19:00       40
...
23:00       48
```

Daily output:

```text
date          pm25_mean   hours   pm25_until_17   valid
──────────────────────────────────────────────────────
01 Mar 2026      83.6       23          84.2        True
```

Many hourly rows have become one daily row.

---

### The 18-hour validity rule

A day is considered valid only when it has at least 18 hourly readings:

```text
hours < 18  → invalid day
hours >= 18 → valid day
```

Visual boundary:

```text
17 readings → ❌ invalid

● ● ● ● ● ● ● ● ● ● ● ● ● ● ● ● ●
                 only 17 hours


18 readings → ✅ valid

● ● ● ● ● ● ● ● ● ● ● ● ● ● ● ● ● ●
                 18 hours
```

This prevents the model from learning from a daily average based on too little data.

Example:

```text
Day A: 23 readings → valid
Day B: 18 readings → valid
Day C: 17 readings → invalid
```

The number `18` is a project rule, not a mathematical truth.

It is the minimum evidence DAF-09 decided was acceptable.

---

### India midnight must define the day

The data uses India time, `+05:30`.

Therefore, a new day starts at:

```text
00:00 India time
```

not at:

```text
00:00 UTC
```

Visual comparison:

```text
India time                         UTC time
──────────────────────             ──────────────────────
25 Mar 23:30 +05:30  ───────────► 25 Mar 18:00 UTC
26 Mar 00:00 +05:30  ───────────► 25 Mar 18:30 UTC
```

The same instant belongs to:

```text
26 March in India
25 March in UTC
```

If we group by UTC instead of India time, one Indian day is split incorrectly.

Correct:

```text
25 Mar India: 00:00 ... 23:00
26 Mar India: 00:00 ... 23:00
```

Wrong UTC-based grouping can move the early-morning India readings into the previous day.

That changes:

- `pm25_mean`,
- `hours`,
- `valid`,
- and eventually the forecast.

---

### What does `pm25_until_17` mean?

At 18:00, the service should use only the hours already known:

```text
00:00 ─────────────────────── 17:00 │ 18:00 ─────── 23:00
       information available by 18:00│ future evening
```

So:

```text
pm25_until_17 = mean(hours 00:00 through 17:00)
```

It must not use:

```text
18:00, 19:00, ..., 23:00
```

Visual example:

```text
Hours used:

00  01  02  03  ...  16  17
 ●   ●   ●   ●       ●   ●
 └────────── average ────────┘

Hours not used:

18  19  20  21  22  23
 ○   ○   ○   ○   ○   ○
```

This prevents the feature from using information that would not exist when the live forecast runs.

---

### Why `hours_known_at_18()` is needed

A value can appear at 17:00 but still be an estimate that needed a future value.

Example:

```text
16:00 → 80
17:00 → missing
18:00 → 100
```

If interpolation fills 17:00:

```text
16:00 → 80
17:00 → 90   ← this used the 18:00 value
18:00 → 100
```

Then the 17:00 value was not truly known at 18:00.

For `pm25_until_17`, that filled value must be excluded:

```text
Raw hourly values       Values known at 18:00
─────────────────       ────────────────────
16:00 → 80              16:00 → 80
17:00 → 90               17:00 → NaN
18:00 → 100              18:00 → not included anyway
```

The rule is:

```text
A value may be used only if it could have been known at forecast time.
```

This protects the model from looking into the future during training.

---

### `add_target()`: move tomorrow's answer onto today's row

After `to_daily()`, we have daily facts but no target:

```text
date          pm25_mean   target
────────────────────────────────
01 Mar 2026      83.6       ?
02 Mar 2026      91.2       ?
03 Mar 2026      76.4       ?
```

The model sees today's features and must predict tomorrow's mean.

Therefore:

```text
01 Mar features → target = 02 Mar mean
02 Mar features → target = 03 Mar mean
03 Mar features → target = 04 Mar mean
```

Visual movement:

```text
Daily means:

01 Mar → 83.6
02 Mar → 91.2
03 Mar → 76.4
04 Mar → 88.0


After add_target():

date          today's features     tomorrow's target
────────────────────────────────────────────────────
01 Mar        83.6                 91.2
02 Mar        91.2                 76.4
03 Mar        76.4                 88.0
04 Mar        88.0                 NaN
```

The final day has no target because tomorrow's observed value is not available yet.

---

### The dangerous `shift()` mistake

Correct:

```text
today's row ← tomorrow's mean
```

```text
01 Mar row ← 02 Mar mean
02 Mar row ← 03 Mar mean
```

Incorrect:

```text
today's row ← yesterday's mean
```

```text
01 Mar row ← 28 Feb mean
02 Mar row ← 01 Mar mean
```

Both versions produce a table.

Nothing crashes.

But the model would be trained against the wrong answer.

That is why `add_target()` owns one explicit rule:

```text
target = tomorrow's pm25_mean
```

---

### The complete visual picture

```text
┌──────────────────────────────────────────────┐
│ Step 3 output: cleaned hourly data            │
│                                              │
│ 00:00, 01:00, 02:00, ... 23:00              │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│ to_daily()                                   │
│                                              │
│ group by India date                          │
│ count valid hours                            │
│ calculate daily averages                     │
│ apply the 18-hour validity rule               │
│ keep only information available by 18:00     │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│ Daily table without target                   │
│                                              │
│ date | pm25_mean | hours | until_17 | valid  │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│ add_target()                                 │
│                                              │
│ copy tomorrow's pm25_mean to today's target  │
│ leave target empty if tomorrow is invalid    │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│ Final training table                         │
│                                              │
│ today's features → tomorrow's PM2.5 target   │
└──────────────────────────────────────────────┘
```

---

### What the real dataset shows

The module reproduces the DAF-09 tables exactly:

```text
Hourly cleaned data
        │
        ▼
to_daily()
        │
        ▼
571 daily rows
        │
        ▼
add_target()
        │
        ▼
daily table with tomorrow's target
```

The important result is not only the row count.

The functions preserve the project rules:

```text
✅ India midnight defines each day
✅ 18 hours is the validity boundary
✅ evening hours are excluded from pm25_until_17
✅ future-dependent fills are not used in the 18:00 feature
✅ target means tomorrow, not yesterday
✅ invalid tomorrow → no target
```

---

### Final takeaway

Step 4 changes:

```text
hourly observations
        ↓
one trustworthy row per India day
        ↓
today's features paired with tomorrow's answer
```

So the final table tells the model:

```text
Given everything known about today by 18:00,
what will tomorrow's average PM2.5 be?
```

## Step 4. `to_daily()` and `add_target()`: hourly → one row per day

### 😖 The problem

DAF-09 built the daily table with one function, `build_daily()`, that did
two different jobs at once: squash a day's hours into `pm25_mean`,
`hours`, `pm25_until_17`, `valid` — **and** reach into tomorrow's row to
set `target`. That's fine in a notebook, but it hides a real risk: the
`target` line is exactly the kind of code a `shift()` typo breaks
silently. DAF-05 shipped that very bug once — `shift(1)` where it needed
`shift(-1)`, so every row's target was **yesterday's** mean instead of
**tomorrow's**. Nothing crashed; the model just trained on the wrong
answer.

Splitting the two jobs makes that bug easy to test on its own, without
needing a whole cleaned dataset to set up. `to_daily()` builds the
per-day columns; `add_target()` is the one small function that owns the
`shift(-1)` rule.

`to_daily()` also carries DAF-09's one subtlety: `pm25_until_17` must not
use a fill that needed a reading from 18:00 or later, because that's a
number Asha doesn't have yet at 18:00. That rule now lives in its own
helper, `hours_known_at_18()`, run on `hourly_raw` (before cleaning) and
`hourly_clean` (after) — the only place either the module or its tests
touch both hourly series at once.

### 💡 The fix

```text
hourly_clean, hourly_raw (Step 2/3)
    │
    ▼
┌──────────────────── to_daily() ────────────────────┐
│  hours_known_at_18()                                │
│    → drop fills that needed an evening reading,     │
│      for pm25_until_17 only                         │
│  resample("D"): pm25_mean, hours, pm25_until_17     │
│  valid = hours >= min_hours                         │
│  reindex to a gap-free calendar                     │
└──────────────────────────────────────────────────────┘
    │
    ▼
daily (one row per day, no target yet)
    │
    ▼  add_target()
    │    target = tomorrow's pm25_mean, only if tomorrow is valid
    ▼
daily with target
```


In [8]:
# Step 4a: the daily table, checked against DAF-09's daily_17.csv (uncleaned) and daily_17_clean.csv
from delhi_air.clean import add_target, to_daily

daily_before = add_target(to_daily(hourly, hourly_raw=hourly))
daily_clean = add_target(to_daily(hourly_clean, hourly_raw=hourly))

expected_before = pd.read_csv(interim_dir / "daily_17.csv", parse_dates=["date"], index_col="date")
expected_clean = pd.read_csv(interim_dir / "daily_17_clean.csv", parse_dates=["date"], index_col="date")

# The recipe on UNCLEANED hours must reproduce DAF-05/DAF-09's original table exactly
pd.testing.assert_frame_equal(daily_before, expected_before, check_names=False, check_freq=False)
print("✅ to_daily(uncleaned) matches daily_17.csv exactly:", daily_before.shape)

# ...and on cleaned hours, DAF-09's cleaned table exactly
pd.testing.assert_frame_equal(daily_clean, expected_clean, check_names=False, check_freq=False)
print("✅ to_daily(cleaned) matches daily_17_clean.csv exactly:", daily_clean.shape)

print("\nValid days:", int(daily_before["valid"].sum()), "→", int(daily_clean["valid"].sum()))
POOR = 91
print("Poor days (pm25_mean >= 91):", int((daily_before["pm25_mean"] >= POOR).sum()),
      "→", int((daily_clean["pm25_mean"] >= POOR).sum()))

✅ to_daily(uncleaned) matches daily_17.csv exactly: (571, 5)
✅ to_daily(cleaned) matches daily_17_clean.csv exactly: (571, 5)

Valid days: 488 → 488
Poor days (pm25_mean >= 91): 173 → 168


### Step 4 tests: the four rules a matching row count can't prove

`daily_clean` matching `daily_17_clean.csv` proves the module gets the
same answer as DAF-09 **on this dataset, today**. It says nothing about
*why* — four separate rules had to each be right, and a test exists for
each one:

- **17 hours is invalid, 18 is valid.** The `min_hours` cutoff, checked at
  its exact boundary.
- **The day boundary is IST midnight, not UTC midnight.** 26 March 00:00
  IST is still 25 March in UTC (18:30). A day-grouping bug that used naive
  UTC dates would put that hour on the wrong day — this is the trap the
  ticket calls out by name.
- **`pm25_until_17` ignores hours 18–23.** Even when those hours hold
  wildly different numbers (1000 vs 10 in the test), they must not move
  `pm25_until_17`.
- **`hours_known_at_18()` drops an evening-dependent fill, keeps an
  earlier one.** Two holes of the same shape, one at 17:00 (needs the
  18:00 reading) and one at 10:00 (doesn't) — only the first is masked
  out.

And for `add_target()`, the two tests the ticket's "explain back" question
is really asking about:

- **Target is tomorrow's mean, on today's row** — checked against the
  *specific* neighbouring value, not just "not null", so a `shift(1)` vs
  `shift(-1)` mix-up fails loudly instead of passing by accident.
- **Target is empty when tomorrow is invalid or missing entirely** — the
  exact DAF-05 leak this function exists to prevent.

In [9]:
# Step 4b: run the tests
import sys
!cd {project_root} && {sys.executable} -m pytest tests/test_clean.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.14, pytest-8.3.3, pluggy-1.6.0 -- /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast
configfile: pyproject.toml
plugins: anyio-4.15.1
collecting ... 


collected 19 items                                                             

tests/test_clean.py::test_load_raw_keeps_only_pm25_sorted_and_in_ist 

PASSED [  5%]
tests/test_clean.py::test_load_raw_fails_loudly_when_station_has_no_files PASSED [ 10%]
tests/test_clean.py::test_to_hourly_averages_readings_inside_the_same_hour PASSED [ 15%]
tests/test_clean.py::test_to_hourly_leaves_a_gap_as_nan_instead_of_skipping_it PASSED [ 21%]
tests/test_clean.py::test_to_hourly_does_not_require_sorted_input PASSED [ 26%]
tests/test_clean.py::test_fill_short_holes_fills_up_to_the_limit_and_no_further PASSED [ 31%]
tests/test_clean.py::test_remove_days_blanks_only_the_named_range PASSED [ 36%]
tests/test_clean.py::test_apply_cleaning_removes_the_broken_week_before_filling_holes PASSED [ 42%]
tests/test_clean.py::test_apply_cleaning_still_fills_short_holes_outside_the_broken_week PASSED [ 47%]
tests/test_clean.py::test_apply_cleaning_does_not_change_its_input PASSED [ 52%]
tests/test_clean.py::test_to_daily_17_hours_is_invalid_18_is_valid PASSED [ 57%]
tests/test_clean.py::test_to_daily_day_boundary_is_ist_midnight_not_utc PASSED [ 63%]
tests/test_

PASSED [ 94%]
tests/test_clean.py::test_main_fails_loudly_when_the_station_has_no_raw_files PASSED [100%]

============================== 19 passed in 0.32s ==============================


### What Step 4 shows

- `to_daily()` + `add_target()` reproduce **both** DAF-09 tables exactly —
  `daily_17.csv` (571 rows, uncleaned) and `daily_17_clean.csv` (571 rows,
  cleaned) — same shape, same values, checked with
  `pd.testing.assert_frame_equal`, not eyeballed.
- **Valid days: 488 → 488.** Filling small holes made 7 days valid;
  removing the broken week made 7 different days invalid. Same count,
  different days.
- **Poor days: 173 → 168.** Matches DAF-09's finding: cleaning didn't
  delete a real bad-air day, it removed 4 fake ones from the broken week
  plus made 1 day more honest.
- **17 tests pass**, 7 of them new — including the UTC/IST boundary test
  the ticket names as a trap, and the two `shift(-1)` tests for
  `add_target()`.
- `build_daily()`'s one job is now two small, separately-tested functions.
  Nothing in this notebook builds the daily table by hand any more.

**Step 4 finishes the pipeline this ticket set out to move**: raw files →
`load_raw()` → `to_hourly()` → `apply_cleaning()` → `to_daily()` →
`add_target()`, all in `clean.py`, all covered by `tests/test_clean.py`.
What's left for DAF-10: the CLI entry point and pointing DAF-09 at the
module instead of its own copy.

## Step 5: A command-line entry point

### 🎬 How we got here

In Steps 1–4, the pipeline worked correctly:

```text
load_raw()
    ↓
to_hourly()
    ↓
apply_cleaning()
    ↓
to_daily()
    ↓
add_target()
```

However, we had to run each step manually inside the notebook.

A live forecasting service cannot open this notebook every day at 18:00. It needs one repeatable command that performs the same pipeline automatically.

---

### 😖 The problem

The future service needs to do this:

```text
Station 17
    │
    ▼
Read raw files
    │
    ▼
Clean and aggregate the data
    │
    ▼
Create the daily table
    │
    ▼
Save daily_17.parquet
```

If the service copies the notebook code, we may create two different implementations:

```text
Notebook:
India time → correct daily table

Service copy:
UTC time → different daily table
```

Nothing may crash, but the model will receive data prepared differently from its training data.

This is called **training/serving skew**.

---

### 💡 The idea

Put the complete pipeline behind one reusable function:

```text
build_daily_table()
```

Then expose it through one terminal command:

```bash
python -m delhi_air.clean --station 17
```

This is similar to a backend service endpoint:

```text
Command-line request
        │
        ▼
main()
        │
        ▼
build_daily_table()
        │
        ▼
daily_17.parquet
```

The notebook and the future live service can both use the same implementation.

---

### 🔍 What the command actually does

```text
$ python -m delhi_air.clean --station 17
                         │
                         ▼
                 main() reads station 17
                         │
                         ▼
┌─────────────────────────────────────────────┐
│ build_daily_table()                         │
│                                             │
│ 1. load_raw()                               │
│ 2. to_hourly()                              │
│ 3. apply_cleaning()                         │
│ 4. to_daily()                               │
│ 5. add_target()                             │
└──────────────────────┬──────────────────────┘
                       │
                       ▼
          data/processed/daily_17.parquet
```

The command is not a second implementation of the pipeline.

It is only another way to call the same functions already tested in `clean.py`.

---

### The complete input-to-output flow

```text
Raw compressed files
        │
        ▼
┌──────────────────────┐
│ load_raw()           │
│ Keep PM2.5 readings  │
└──────────┬───────────┘
           ▼
┌──────────────────────┐
│ to_hourly()          │
│ 15-minute → hourly   │
└──────────┬───────────┘
           ▼
┌──────────────────────┐
│ apply_cleaning()     │
│ Remove bad periods   │
│ Fill short gaps      │
└──────────┬───────────┘
           ▼
┌──────────────────────┐
│ to_daily()           │
│ Hourly → daily rows  │
└──────────┬───────────┘
           ▼
┌──────────────────────┐
│ add_target()         │
│ Attach tomorrow's    │
│ PM2.5 mean           │
└──────────┬───────────┘
           ▼
┌──────────────────────┐
│ Parquet output       │
│ daily_17.parquet     │
└──────────────────────┘
```

---

### What `--station 17` means

The argument identifies the station whose files should be processed:

```text
--station 17
      │
      ▼
data/raw/openaq/locationid=17/
```

The command searches inside that station directory, reads all raw files, and creates:

```text
data/processed/daily_17.parquet
```

The output contains one row per India calendar day:

```text
date          pm25_mean   hours   valid   target
────────────────────────────────────────────────
01 Mar 2026      83.6       23    True     91.2
02 Mar 2026      91.2       24    True     76.4
03 Mar 2026      76.4       17    False     NaN
```

---

### Why `--raw-root` and `--out-dir` exist

The command has optional folder arguments:

```text
--raw-root
    Where the raw station files are located

--out-dir
    Where the generated parquet file should be written
```

Normally, the command uses:

```text
data/raw/
data/processed/
```

Tests override these paths with temporary folders:

```text
Temporary raw folder
        │
        ▼
CLI
        │
        ▼
Temporary output folder
```

This allows the tests to use a few fake readings instead of the real 44,139-row dataset.

---

### 🤔 Before you run Cell 5a

The notebook pipeline already produced `daily_clean`.

The CLI runs the same five functions.

What do you think will happen?

```text
(a) The CLI will produce a different table because it runs outside the notebook.

(b) The CLI will produce the same 571-row table because it calls the same pipeline.

(c) The CLI will create an empty file because no notebook variables are available.
```

Run Cell 5a and check your prediction.

---

### What Cell 5a checks

Cell 5a starts the CLI as a separate operating-system process:

```text
Notebook
   │
   ├── starts python -m delhi_air.clean --station 17
   │
   ├── waits for the process to finish
   │
   └── checks the generated parquet file
```

It verifies four things:

```text
1. The command exits successfully.
2. daily_17.parquet is created.
3. The parquet file can be read.
4. The CLI table equals the notebook table exactly.
```

The final comparison is:

```text
Notebook result == CLI result
```

This is important because a command that merely creates a file is not enough. The file must contain the same data as the tested notebook pipeline.

---

### Visual comparison: notebook versus CLI

```text
Notebook pipeline                    CLI pipeline
─────────────────                    ────────────
load_raw()                           load_raw()
to_hourly()                          to_hourly()
apply_cleaning()       ==            apply_cleaning()
to_daily()                            to_daily()
add_target()                         add_target()
      │                                    │
      ▼                                    ▼
daily_clean                          daily_17.parquet
```

Both paths should produce:

```text
571 rows
same columns
same values
same targets
```

---

### ✅ What Cell 5a should show

If everything works, the output should look similar to:

```text
✅ CLI wrote 571 rows to data/processed/daily_17.parquet,
identical to the notebook's daily_clean
```

This means:

```text
The command completed successfully
        +
The output file exists
        +
The file contains 571 daily rows
        +
The CLI output matches the notebook output
```

---

### CLI tests in Cell 5b

After checking the real station, Cell 5b runs the complete test suite again:

```text
Step 1 tests → load_raw()
Step 2 tests → to_hourly()
Step 3 tests → apply_cleaning()
Step 4 tests → to_daily() and add_target()
Step 5 tests → command-line pipeline
```

The two new Step 5 tests use a tiny temporary station.

---

### Test 1: the CLI happy path

```text
Tiny fake station
        │
        ▼
main()
        │
        ▼
build_daily_table()
        │
        ▼
temporary daily_17.parquet
        │
        ▼
Read the file and check its contents
```

This proves that the complete pipeline can run from start to finish without notebook variables.

---

### Test 2: missing station fails loudly

```text
Station 999
    │
    └── no raw files
            │
            ▼
      FileNotFoundError
            │
            ▼
         Test passes
```

The unsafe behaviour would be:

```text
No files
   │
   ▼
Empty table
   │
   ▼
Empty output
   │
   ▼
Nobody notices
```

The safe behaviour is:

```text
No files
   │
   ▼
Clear error
   │
   ▼
The problem is noticed immediately
```

---

### ✅ Checkpoint

1. Why must the CLI use the same functions as the notebook?

2. What should happen when the requested station has no raw files?

3. Why do the CLI tests use temporary data instead of the real dataset?

---

### Answers

1. The same functions prevent training/serving skew. The service must prepare data exactly as the training pipeline did.

2. It should raise `FileNotFoundError` instead of silently creating an empty output.

3. Temporary data makes the tests fast, repeatable, and independent of the large real dataset.

---

### What Step 5 proves

```text
✅ The pipeline runs without opening the notebook
✅ The CLI uses the same functions as the notebook
✅ The result is saved as Parquet
✅ The CLI output matches the notebook output
✅ Missing input data produces an immediate error
✅ The complete suite reaches 19 passing tests
```

### Final takeaway

Step 5 changes the workflow from:

```text
A person manually running notebook cells
```

to:

```text
One repeatable command:

python -m delhi_air.clean --station 17
```

The notebook is now useful for learning and verification.

The command-line entry point is ready for a scheduler, Docker container, cron job, or future live forecasting service.

## Step 5. A command-line entry point

### 😖 The problem

Every step so far runs inside a notebook, one cell at a time, with a human
watching. Phase 8's live service can't do that — nothing opens a notebook
at 18:00 every evening. It needs one command it can call from a cron job,
a scheduler, or a Docker container: give it a station, get back today's
row, no notebook involved.

That command also has to be the **exact same pipeline** as the notebook —
`load_raw` → `to_hourly` → `apply_cleaning` → `to_daily` → `add_target` —
in the same order, with the same settings. If the CLI drifted even slightly
from what trained the model, that's training/serving skew again, just
moved one level up.

### 💡 The fix

`build_daily_table()` wraps the five functions in one call, and `main()`
turns that into a command:

```text
$ python -m delhi_air.clean --station 17
                │
                ▼
┌───────────────── build_daily_table() ─────────────────┐
│  load_raw → to_hourly → apply_cleaning → to_daily      │
│  → add_target                                          │
│  (exactly the calls Steps 1-4 made, in the same order) │
└──────────────────────────────────────────────────────┘
                │
                ▼
   data/processed/daily_17.parquet
```

Two flags, `--raw-root` and `--out-dir`, override the default `data/`
folders — that's what lets the CLI's own tests point at a tiny, temporary
folder instead of the real 44,139-reading dataset.

In [10]:
# Step 5a: run the real CLI as a subprocess, exactly as Phase 8 would call it
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "delhi_air.clean", "--station", str(STATION)],
    cwd=project_root, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0, "the CLI exited with an error"

out_path = project_root / "data/processed/daily_17.parquet"
assert out_path.exists(), "the CLI did not write the parquet file"

# The CLI's output must be identical to Step 4's in-notebook result
from_cli = pd.read_parquet(out_path)
from_cli.index = from_cli.index.tz_convert(daily_clean.index.tz)   # parquet round-trips the tz object, not just the offset
pd.testing.assert_frame_equal(from_cli, daily_clean, check_names=False, check_freq=False)
print(f"\n✅ CLI wrote {len(from_cli)} rows to {out_path.relative_to(project_root)}, identical to the notebook's daily_clean")

Wrote 571 rows to /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/data/processed/daily_17.parquet


✅ CLI wrote 571 rows to data/processed/daily_17.parquet, identical to the notebook's daily_clean


### Step 5 tests: the CLI on a tiny station, not the real one

The subprocess run above proves the CLI works today, on the real 44,139
readings — but it's slow (it re-reads every raw file) and it can't run
anywhere without that data. The two CLI tests use the same trick as every
earlier step: a handful of hand-built rows in a temporary folder.

**The happy path.** Two days of hourly readings, written the way
`load_raw()` expects them. `main()` is called directly (not as a
subprocess — faster, and `pytest` can still see a real exception if one
is raised) with `--raw-root` and `--out-dir` pointing at `tmp_path`. The
test then reads the parquet back and checks the columns and row count —
proof that every step of the pipeline actually ran, not just that a file
appeared.

**Fail loudly on a missing station.** `main()` for a station with no raw
files must raise `FileNotFoundError`, the same failure `load_raw()` raises
on its own — the CLI doesn't swallow it into a silent empty output.

In [11]:
# Step 5b: run the tests
!cd {project_root} && {sys.executable} -m pytest tests/test_clean.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.14, pytest-8.3.3, pluggy-1.6.0 -- /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast
configfile: pyproject.toml
plugins: anyio-4.15.1
collecting ... 


collected 19 items                                                             

tests/test_clean.py::test_load_raw_keeps_only_pm25_sorted_and_in_ist 

PASSED [  5%]
tests/test_clean.py::test_load_raw_fails_loudly_when_station_has_no_files PASSED [ 10%]
tests/test_clean.py::test_to_hourly_averages_readings_inside_the_same_hour PASSED [ 15%]
tests/test_clean.py::test_to_hourly_leaves_a_gap_as_nan_instead_of_skipping_it PASSED [ 21%]
tests/test_clean.py::test_to_hourly_does_not_require_sorted_input PASSED [ 26%]
tests/test_clean.py::test_fill_short_holes_fills_up_to_the_limit_and_no_further PASSED [ 31%]
tests/test_clean.py::test_remove_days_blanks_only_the_named_range PASSED [ 36%]
tests/test_clean.py::test_apply_cleaning_removes_the_broken_week_before_filling_holes PASSED [ 42%]
tests/test_clean.py::test_apply_cleaning_still_fills_short_holes_outside_the_broken_week PASSED [ 47%]
tests/test_clean.py::test_apply_cleaning_does_not_change_its_input PASSED [ 52%]
tests/test_clean.py::test_to_daily_17_hours_is_invalid_18_is_valid PASSED [ 57%]
tests/test_clean.py::test_to_daily_day_boundary_is_ist_midnight_not_utc PASSED [ 63%]
tests/test_

PASSED [ 94%]
tests/test_clean.py::test_main_fails_loudly_when_the_station_has_no_raw_files PASSED [100%]

============================== 19 passed in 0.40s ==============================


### What Step 5 shows

- `python -m delhi_air.clean --station 17` writes **571 rows** to
  `data/processed/daily_17.parquet`, byte-for-byte the same table Step 4
  built in the notebook.
- **19 tests pass**, 2 of them new: the CLI's happy path and its
  fail-loudly case.
- `data/processed/` now holds the parquet a live service would read; `git
  status` shows only code and tests changing — `data/` stays out of git,
  as it always has.

**DAF-10 is now fully in `clean.py`:** raw files in, a cleaned daily table
with a target column out, callable from Python or from the command line,
with 19 tests covering every rule along the way. What's left on the
ticket: pointing the DAF-09 notebook itself at these functions, so no
notebook keeps its own copy of the table-building logic.